#  04 Homework 04 ETM

## 1. Import the needed libraries

In [ ]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Graph import Graph
from topologicpy.Helper import Helper

## 2. Check the TopologicPy version

In [ ]:
print("This tutorial requires topologicpy version 0.9.18 or newer.")
print(Helper.Version())

## 3. Set your renderer
* Visual Studio Code: `"vscode"`
* Google Colab: `"colab"`
* Browser: `"browser"`

In [ ]:
renderer = "vscode"

In [ ]:
import os
os.makedirs("Images", exist_ok=True)

## 4. Import room OBJs by type

In [ ]:
BASE = r"C:\Users\etmaglari\IAAC\etmaglari_gML\Homework04\Objects"

ROOM_TYPES = {
    "Bedroom":     {"path": BASE + r"\Bedroom.obj",     "color": "#E63946"},
    "Bathroom":    {"path": BASE + r"\Bathroom.obj",    "color": "#4CC9F0"},
    "Corridor":    {"path": BASE + r"\Corridor.obj",    "color": "#457B9D"},
    "Kitchen":     {"path": BASE + r"\Kitchen.obj",     "color": "#F4A261"},
    "Living room": {"path": BASE + r"\Living room.obj", "color": "#FFBE0B"},
    "Stair":       {"path": BASE + r"\Stair.obj",       "color": "#8338EC"},
}

def dist3(a, b):
    return ((a[0]-b[0])**2 + (a[1]-b[1])**2 + (a[2]-b[2])**2)**0.5

def get_val(topology, key):
    d = Topology.Dictionary(topology)
    if d is None:
        return None
    v = Dictionary.ValueAtKey(d, key)
    if isinstance(v, list):
        return v[0] if v else None
    return v

def try_build_cell(faces):
    """Try Cell.ByFaces at progressively looser tolerances; return first success."""
    for face_set in [faces, faces]:  # try same set; different tols
        for tol in [0.001, 0.005, 0.01, 0.05, 0.1]:
            c = Cell.ByFaces(face_set, tolerance=tol)
            if c is not None:
                return c
    return None

all_faces_raw   = []
all_faces_clean = []
selector        = []
cells           = []

for type_name, info in ROOM_TYPES.items():
    objs = Topology.ByOBJPath(info["path"], selfMerge=False)
    if not isinstance(objs, list):
        objs = [objs]

    count = 0
    for i, obj in enumerate(objs):
        raw_faces = Topology.Faces(obj) or []
        if not raw_faces:
            continue
        all_faces_raw.extend(raw_faces)

        cleaned_obj = Topology.RemoveCoplanarFaces(obj, epsilon=0.1, tolerance=0.001, silent=True)
        clean_obj   = cleaned_obj if cleaned_obj else obj
        clean_faces = Topology.Faces(clean_obj) or []
        all_faces_clean.extend(clean_faces)

        d = Dictionary.ByKeysValues(
            ["color", "type", "label", "vertex_size"],
            [info["color"], type_name, type_name, 20]
        )

        # Try raw faces first, then cleaned, at multiple tolerances
        c = try_build_cell(raw_faces) or try_build_cell(clean_faces)

        if c is not None:
            c2 = Topology.RemoveCoplanarFaces(c, epsilon=0.1, tolerance=0.001, silent=True)
            c  = c2 if c2 else c
            c  = Topology.RemoveCollinearEdges(c) or c
            s  = Topology.InternalVertex(c)
            c  = Topology.SetDictionary(c, d)
            cells.append(c)
        else:
            # Centroid fallback so the selector always exists
            cen = Topology.Centroid(Cluster.ByTopologies(raw_faces))
            s   = Vertex.ByCoordinates(cen.X(), cen.Y(), cen.Z())
            print(f"  fallback selector: {type_name} #{i+1}")

        selector.append(Topology.SetDictionary(s, d))
        count += 1

    print(f"{type_name}: {count} object(s)  |  cells so far: {len(cells)}")

print(f"\nTotal selectors : {len(selector)}")
print(f"Total cells     : {len(cells)}")
print(f"Raw faces       : {len(all_faces_raw)}")

## 5. Show the geometry

In [ ]:
Topology.Show(
    cells,
    selector,
    faceColorKey="color",
    faceOpacity=0.4,
    showEdges=True,
    edgeWidth=3,
    showVertices=True,
    vertexSize=15,
    vertexLabelKey="label",
    showVertexLabel=True,
    backgroundColor="black",
    width=800,
    height=600,
    renderer=renderer
)

## 6. Compute adjacency from face-centroid overlap

In [ ]:
from collections import Counter

# ── Face centroid helpers ────────────────────────────────────────────────────
def face_cent_tuple(face, r=2):
    c = Topology.Centroid(face)
    return (round(Vertex.X(c), r), round(Vertex.Y(c), r), round(Vertex.Z(c), r))

# Per-cell: list of face centroids (for intersection), and face-plane data for open-space
cell_face_cents  = []
cell_face_planes = []           # (centroid_xyz, unit_normal)  — reused in sec 13
for c in cells:
    fcts = []
    fpls = []
    for f in (Topology.Faces(c) or []):
        fc = Topology.Centroid(f)
        fp = (Vertex.X(fc), Vertex.Y(fc), Vertex.Z(fc))
        fn = Face.Normal(f)
        fcts.append(face_cent_tuple(f))
        fpls.append((fp, fn))
    cell_face_cents.append(fcts)
    cell_face_planes.append(fpls)

cell_face_sets = [frozenset(x) for x in cell_face_cents]

# ── Adjacency: cells sharing a face centroid ─────────────────────────────────
adj_pairs = []
for i in range(len(cells)):
    for j in range(i + 1, len(cells)):
        if cell_face_sets[i] & cell_face_sets[j]:
            adj_pairs.append((i, j))

# ── Open-space adjacency: wall faces co-planar & opposite-normal, < 1 m apart ─
# Catches rooms with open boundaries (no wall) missed by exact centroid matching
def dot3(a, b): return a[0]*b[0] + a[1]*b[1] + a[2]*b[2]
def perp_dist(pt, fp, fn):
    dx, dy, dz = pt[0]-fp[0], pt[1]-fp[1], pt[2]-fp[2]
    return abs(fn[0]*dx + fn[1]*dy + fn[2]*dz)
def is_horiz(fn): return abs(fn[2]) > 0.7   # floor/ceiling face, skip

adj_set = set(adj_pairs)
open_pairs = []
for i in range(len(cells)):
    for j in range(i + 1, len(cells)):
        if (i, j) in adj_set:
            continue
        found = False
        for fp_i, fn_i in cell_face_planes[i]:
            if is_horiz(fn_i):
                continue
            for fp_j, fn_j in cell_face_planes[j]:
                if is_horiz(fn_j):
                    continue
                if dot3(fn_i, fn_j) > -0.85:      # must be roughly opposite normals
                    continue
                if perp_dist(fp_i, fp_j, fn_j) >= 0.1:  # must be in same plane
                    continue
                if dist3(fp_i, fp_j) < 1.0:        # face centroids within 1 m
                    open_pairs.append((i, j))
                    found = True
                    break
            if found:
                break

# ── Area-scaled vertex sizes ─────────────────────────────────────────────────
areas    = [Cell.SurfaceArea(c) for c in cells]
min_area = min(areas)
max_area = max(areas)

for i, (s, area) in enumerate(zip(selector, areas)):
    size  = 12 + int(48 * (area - min_area) / (max_area - min_area + 1))
    d_old = Topology.Dictionary(s)
    keys  = list(Dictionary.Keys(d_old)) + ["area", "vertex_size"]
    vals  = list(Dictionary.Values(d_old)) + [area, size]
    Topology.SetDictionary(s, Dictionary.ByKeysValues(keys, vals))

cell_cluster = Cluster.ByTopologies(cells)

print(f"Individual cells  : {len(cells)}  (expected 22)")
print(f"Adjacent pairs    : {len(adj_pairs)}")
print(f"Open-space pairs  : {len(open_pairs)}")

types_found = Counter(get_val(s, "type") or "unknown" for s in selector)
print("\nRoom type breakdown:")
for t, n in sorted(types_found.items()):
    print(f"  {t}: {n}")

## 7. Primal Graph

Vertices at the geometric corners of the CellComplex; edges along wall boundaries.

In [ ]:
primal_verts = Topology.Vertices(cell_cluster) or []
primal_edges = Topology.Edges(cell_cluster) or []
print(f"Primal — Vertices: {len(primal_verts)}, Edges: {len(primal_edges)}")

Topology.Show(
    cell_cluster,
    faceColor=[210, 210, 250], faceOpacity=0.15,
    edgeColor="red", edgeWidth=2,
    vertexColor="red", vertexSize=6,
    showVertices=True,
    backgroundColor="black",
    width=800, height=600,
    renderer=renderer
)

## 8. Dual Graph

One vertex per room (cell centroid); an edge between every pair of rooms that share a wall face.

In [ ]:
# One edge per adjacent pair — selector vertices are the nodes
adj_edges = [Edge.ByVertices([selector[i], selector[j]]) for i, j in adj_pairs]
g_dual = Graph.ByVerticesEdges(selector, adj_edges)
print(f"Dual — Nodes: {len(Graph.Vertices(g_dual))}, Edges: {len(Graph.Edges(g_dual))}")

Topology.Show(
    g_dual, cell_cluster,
    faceColorKey="color",
    faceOpacity=0.05,
    edgeColor="white", edgeWidth=0.5,
    vertexSizeKey="vertex_size",
    vertexColorKey="color",
    vertexLabelKey="label",
    showVertexLabel=True,
    showVertices=True,
    backgroundColor="black",
    width=800, height=600,
    renderer=renderer
)

## 9. Adjacency Graph (by shared wall)

In [ ]:
# Adjacency graph: same edges as dual, but nodes labelled with room type + area size
g_adj = Graph.ByVerticesEdges(selector, adj_edges)

for v in Graph.Vertices(g_adj) or []:
    vx, vy, vz = Vertex.X(v), Vertex.Y(v), Vertex.Z(v)
    best_s = min(selector, key=lambda s: dist3(
        (vx, vy, vz), (Vertex.X(s), Vertex.Y(s), Vertex.Z(s))
    ))
    color = get_val(best_s, "color")       or "gray"
    label = get_val(best_s, "label")       or ""
    size  = get_val(best_s, "vertex_size") or 14
    if isinstance(size, list):
        size = size[0]
    Topology.SetDictionary(v, Dictionary.ByKeysValues(
        ["color", "label", "size"], [color, label, int(size)]
    ))

for e in Graph.Edges(g_adj) or []:
    Topology.SetDictionary(e, Dictionary.ByKeysValues(["width", "color"], [3, "black"]))

print(f"Adjacency graph — vertices: {len(Graph.Vertices(g_adj))}, edges: {len(Graph.Edges(g_adj))}")

Topology.Show(
    g_adj, cell_cluster,
    faceColorKey="color",
    faceOpacity=0.10,
    showEdges=True,
    edgeWidthKey="width",
    edgeColorKey="color",
    vertexSizeKey="size",
    vertexColorKey="color",
    vertexLabelKey="label",
    showVertexLabel=True,
    showVertices=True,
    backgroundColor="white",
    width=1000, height=1000,
    renderer=renderer
)

## 9. Load Windows

In [ ]:
WIN_PATH = r"C:\Users\etmaglari\IAAC\etmaglari_gML\Homework04\Objects\window.obj"

win_objects = Topology.ByOBJPath(WIN_PATH, selfMerge=False)
if not isinstance(win_objects, list):
    win_objects = [win_objects]

windows = []
for obj in win_objects:
    for w in (Topology.Wires(obj) or []):
        f = Face.ByWire(w)
        if f is not None:
            windows.append(f)

for w in windows:
    Topology.SetDictionary(w, Dictionary.ByKeysValues(["color", "type"], ["#4CC9F0", "window"]))

print(f"Windows found: {len(windows)}")
win_cluster = Cluster.ByTopologies(windows)

Topology.Show(
    cell_cluster, win_cluster,
    faceColor="#4CC9F0", faceOpacity=0.25,
    edgeColor="white", edgeWidth=0.5,
    showVertices=False,
    backgroundColor="black",
    width=800, height=600,
    renderer=renderer
)

## 10. Load Doors

In [ ]:
DOOR_PATH = r"C:\Users\etmaglari\IAAC\etmaglari_gML\Homework04\Objects\door.obj"

door_objects = Topology.ByOBJPath(DOOR_PATH, selfMerge=False)
if not isinstance(door_objects, list):
    door_objects = [door_objects]

doors = []
for obj in door_objects:
    for w in (Topology.Wires(obj) or []):
        f = Face.ByWire(w)
        if f is not None:
            doors.append(f)

for d in doors:
    Topology.SetDictionary(d, Dictionary.ByKeysValues(["color", "type"], ["#F4A261", "door"]))

print(f"Doors found: {len(doors)}")
door_cluster = Cluster.ByTopologies(doors)

Topology.Show(
    cell_cluster, door_cluster,
    faceColor="#F4A261", faceOpacity=0.25,
    edgeColor="white", edgeWidth=0.5,
    showVertices=False,
    backgroundColor="black",
    width=800, height=600,
    renderer=renderer
)

## 11. Load Entrance Doors

In [ ]:
ENTRANCE_PATH = r"C:\Users\etmaglari\IAAC\etmaglari_gML\Homework04\Objects\Entrance door.obj"

entrance_objects = Topology.ByOBJPath(ENTRANCE_PATH, selfMerge=False)
if not isinstance(entrance_objects, list):
    entrance_objects = [entrance_objects]

entrance_doors = []
for obj in entrance_objects:
    for w in (Topology.Wires(obj) or []):
        f = Face.ByWire(w)
        if f is not None:
            entrance_doors.append(f)

for ed in entrance_doors:
    Topology.SetDictionary(ed, Dictionary.ByKeysValues(["color", "type"], ["#E63946", "entrance"]))

print(f"Entrance doors found: {len(entrance_doors)}")
entrance_cluster = Cluster.ByTopologies(entrance_doors)

Topology.Show(
    cell_cluster, entrance_cluster,
    faceColor="#E63946", faceOpacity=0.25,
    edgeColor="white", edgeWidth=0.5,
    showVertices=False,
    backgroundColor="black",
    width=800, height=600,
    renderer=renderer
)

## 12. Primal Graph with Apertures

Overlay the primal graph on the room model and include windows, doors, and entrance doors.

In [ ]:
aperture_cluster = Cluster.ByTopologies(windows + doors + entrance_doors)
primal_verts   = Topology.Vertices(cell_cluster) or []
primal_edges   = Topology.Edges(cell_cluster)    or []
primal_cluster = Cluster.ByTopologies(primal_edges + primal_verts)

print(f"Primal — Vertices: {len(primal_verts)}, Edges: {len(primal_edges)}")

Topology.Show(
    cell_cluster, aperture_cluster, primal_cluster,
    faceColor="#4CC9F0", faceOpacity=0.12,
    edgeColor="red", edgeWidth=2,
    vertexColor="red", vertexSize=6,
    showVertices=True,
    backgroundColor="black",
    width=900, height=700,
    renderer=renderer
)

## 13. Add Apertures to CellComplex

In [ ]:
all_apertures = windows + doors + entrance_doors
all_aperture_types = (
    ["window"]   * len(windows) +
    ["door"]     * len(doors) +
    ["entrance"] * len(entrance_doors)
)
apt_colors = {"window": "#4CC9F0", "door": "#F4A261", "entrance": "#E63946"}

# Centroid of each aperture face
apt_cents = []
for apt in all_apertures:
    c = Topology.Centroid(apt)
    apt_cents.append((Vertex.X(c), Vertex.Y(c), Vertex.Z(c)))

# Extend cell_face_planes with bboxes (cell_face_planes from sec 6 has (fp, fn) only)
cell_face_planes_bbox = []
for c in cells:
    fd = []
    for f in (Topology.Faces(c) or []):
        fc  = Topology.Centroid(f)
        fp  = (Vertex.X(fc), Vertex.Y(fc), Vertex.Z(fc))
        fn  = Face.Normal(f)
        fvs = Topology.Vertices(f) or []
        if fvs:
            xs = [Vertex.X(v) for v in fvs]
            ys = [Vertex.Y(v) for v in fvs]
            zs = [Vertex.Z(v) for v in fvs]
            bbox = (min(xs), max(xs), min(ys), max(ys), min(zs), max(zs))
        else:
            bbox = None
        fd.append((fp, fn, bbox))
    cell_face_planes_bbox.append(fd)

def in_bbox(pt, bbox, margin=0.5):
    if bbox is None:
        return True
    xmn, xmx, ymn, ymx, zmn, zmx = bbox
    return (xmn-margin <= pt[0] <= xmx+margin and
            ymn-margin <= pt[1] <= ymx+margin and
            zmn-margin <= pt[2] <= zmx+margin)

# Aperture is on a room face ↔ plane distance < 0.15 m AND within face bounding box
PLANE_TOL = 0.15
apt_to_rooms = [set() for _ in all_apertures]
for ai, ac in enumerate(apt_cents):
    for ri, fd in enumerate(cell_face_planes_bbox):
        for fp, fn, bbox in fd:
            if perp_dist(ac, fp, fn) < PLANE_TOL and in_bbox(ac, bbox):
                apt_to_rooms[ai].add(ri)
                break

# Access pairs via apertures (rooms connected by a door or window)
access_pairs = set()
for ai, rooms in enumerate(apt_to_rooms):
    if len(rooms) >= 2:
        room_list = sorted(rooms)
        for a in range(len(room_list)):
            for b in range(a + 1, len(room_list)):
                access_pairs.add((room_list[a], room_list[b]))

matched = sum(1 for r in apt_to_rooms if r)
print(f"All apertures: {len(all_apertures)}  "
      f"(windows={len(windows)}, doors={len(doors)}, entrance={len(entrance_doors)})")
print(f"Apertures matched to ≥1 room : {matched}")
print(f"Access pairs via apertures   : {len(access_pairs)}")

aperture_cluster = Cluster.ByTopologies(all_apertures)

Topology.Show(
    cell_cluster, aperture_cluster,
    faceColorKey="color",
    faceOpacity=0.15,
    edgeColor="white", edgeWidth=0.5,
    showVertices=False,
    backgroundColor="black",
    width=800, height=600,
    renderer=renderer
)

## 14. Access + Adjacency Graph

Rooms connected when they **share a wall** (direct=True), share an **interior aperture** (viaSharedApertures=True), or have an **exterior aperture** (toExteriorApertures=True). Node size scales with room surface area.

In [ ]:
# Access graph combines:
#   1. direct adjacency (shared face)  — equiv. to direct=True
#   2. open-space adjacency (no wall)  — e.g. kitchen ↔ living room
#   3. aperture connections (doors/windows between rooms)
all_access_pairs = (
    set(tuple(sorted(p)) for p in adj_pairs)
    | set(tuple(sorted(p)) for p in open_pairs)
    | access_pairs
)
access_edges = [Edge.ByVertices([selector[i], selector[j]]) for i, j in all_access_pairs]
g_access = Graph.ByVerticesEdges(selector, access_edges)

for v in Graph.Vertices(g_access) or []:
    vx, vy, vz = Vertex.X(v), Vertex.Y(v), Vertex.Z(v)
    best_s = min(selector, key=lambda s: dist3(
        (vx, vy, vz), (Vertex.X(s), Vertex.Y(s), Vertex.Z(s))
    ))
    color = get_val(best_s, "color")       or "gray"
    label = get_val(best_s, "label")       or ""
    size  = get_val(best_s, "vertex_size") or 14
    if isinstance(size, list):
        size = size[0]
    Topology.SetDictionary(v, Dictionary.ByKeysValues(
        ["color", "label", "size"], [color, label, int(size)]
    ))

for e in Graph.Edges(g_access) or []:
    Topology.SetDictionary(e, Dictionary.ByKeysValues(["width", "color"], [2, "black"]))

n_adj   = len(set(tuple(sorted(p)) for p in adj_pairs))
n_open  = len(set(tuple(sorted(p)) for p in open_pairs) - set(tuple(sorted(p)) for p in adj_pairs))
n_apt   = len(access_pairs - set(tuple(sorted(p)) for p in adj_pairs) - set(tuple(sorted(p)) for p in open_pairs))
print(f"Access graph — {len(Graph.Vertices(g_access))} nodes, {len(Graph.Edges(g_access))} edges")
print(f"  direct: {n_adj}  |  open-space: {n_open}  |  aperture-only: {n_apt}")

Topology.Show(
    [cell_cluster, g_access, win_cluster, door_cluster, entrance_cluster],
    faceColorKey="color",
    faceOpacity=0.12,
    edgeWidthKey="width",
    edgeColorKey="color",
    vertexSizeKey="size",
    vertexColorKey="color",
    vertexLabelKey="label",
    showVertexLabel=True,
    showVertices=True,
    backgroundColor="white",
    width=1000, height=1000,
    renderer=renderer
)

## 15. Room Centroid to Aperture Bipartite Graph

Three aperture types: **windows** (`#4CC9F0`), **doors** (`#F4A261`), **entrance doors** (`#E63946`). Node size scales with room surface area.

In [ ]:
from collections import defaultdict

apt_colors = {"window": "#4CC9F0", "door": "#F4A261", "entrance": "#E63946"}

# Room nodes — area-scaled size, colour by type
room_verts = []
for i, (c, s) in enumerate(zip(cells, selector)):
    rc = Topology.Centroid(c)
    color = get_val(s, "color") or "gray"
    area  = areas[i]
    size  = 12 + int(48 * (area - min_area) / (max_area - min_area + 1))
    v = Vertex.ByCoordinates(Vertex.X(rc), Vertex.Y(rc), Vertex.Z(rc))
    Topology.SetDictionary(v, Dictionary.ByKeysValues(["size", "color"], [size, color]))
    room_verts.append(v)

# Aperture nodes — one per unique aperture face
aperture_verts = []
for apt, atype in zip(all_apertures, all_aperture_types):
    ac = Topology.Centroid(apt)
    v  = Vertex.ByCoordinates(Vertex.X(ac), Vertex.Y(ac), Vertex.Z(ac))
    Topology.SetDictionary(v, Dictionary.ByKeysValues(
        ["size", "color"], [7, apt_colors[atype]]
    ))
    aperture_verts.append(v)

# Bipartite edges: room ↔ aperture
edges_ra = []
room_to_windows   = defaultdict(int)
room_to_doors     = defaultdict(int)
room_to_entrances = defaultdict(int)

for ai, rooms in enumerate(apt_to_rooms):
    atype  = all_aperture_types[ai]
    ecolor = apt_colors[atype]
    av     = aperture_verts[ai]
    for ri in sorted(rooms):
        rv = room_verts[ri]
        e  = Edge.ByVertices([rv, av])
        Topology.SetDictionary(e, Dictionary.ByKeysValues(["width", "color"], [2, ecolor]))
        edges_ra.append(e)
        if atype == "window":
            room_to_windows[ri] += 1
        elif atype == "door":
            room_to_doors[ri] += 1
        else:
            room_to_entrances[ri] += 1

print("Windows per room:")
for ri in sorted(room_to_windows):
    print(f"  Room {ri} ({get_val(selector[ri],'label')}): {room_to_windows[ri]}")
print("Doors per room:")
for ri in sorted(room_to_doors):
    print(f"  Room {ri} ({get_val(selector[ri],'label')}): {room_to_doors[ri]}")
print("Entrance doors per room:")
for ri in sorted(room_to_entrances):
    print(f"  Room {ri} ({get_val(selector[ri],'label')}): {room_to_entrances[ri]}")
print(f"Bipartite edges: {len(edges_ra)}")

In [ ]:
bipartite = room_verts + aperture_verts + edges_ra

Topology.Show(
    [cell_cluster, aperture_cluster] + bipartite,
    faceColor="#4CC9F0", faceOpacity=0.12,
    edgeColor="white", edgeWidth=0.5,
    vertexColor="black",
    vertexSizeKey="size", vertexColorKey="color",
    edgeWidthKey="width", edgeColorKey="color",
    showVertices=True,
    backgroundColor="black",
    width=900, height=700,
    renderer=renderer
)